In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset


In [ ]:
base_model = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(base_model)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/560 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [ ]:
import zipfile
import os
# Path to your zip file
zip_path = "/content/instructive_folder.zip"

# Extract all files
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall()


In [ ]:
model_path = "/content/content/tinyllama-instruction/checkpoint-3"

In [ ]:
instruction_model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")

model.safetensors:   0%|          | 0.00/4.40G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [ ]:
prompt = "Explain how artificial intelligence is improving the process of drug discovery and development in the pharmaceutical industry."

In [ ]:
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

In [ ]:
outputs = instruction_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

In [ ]:
print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))



Model Output:

Explain how artificial intelligence is improving the process of drug discovery and development in the pharmaceutical industry.
A recent study published in Nature Medicine has found that AI algorithms can better predict which patients with a particular disease will respond to certain drugs than an expert's opinion alone. This has been a significant breakthrough for the pharmaceutical industry, as it can help speed up the development of new medications by providing more accurate results without having to rely on trial-and-error or clinical trials.
What are the advantages of using artificial intelligence (AI) to improve


In [ ]:
!pip install -U trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 17.4 MB/s eta 0:00:00


In [ ]:
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.3 MB/s eta 0:00:00


In [ ]:
!pip install transformers accelerate

In [ ]:
from trl import DPOTrainer
from transformers import AutoTokenizer,  AutoModelForCausalLM, TrainingArguments
from peft import PeftModel
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset
import torch

In [ ]:
base_model = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [ ]:
dataset = load_dataset("csv", data_files="/content/pharma_preference_data.csv")["train"]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(base_model)

In [ ]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none"
)

In [ ]:

base_model

'TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T'

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    device_map="auto",
    quantization_config=None  # or use BitsAndBytesConfig
)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
instruction_model_checkpoint = "/content/content/tinyllama-instruction/checkpoint-3"

In [ ]:
model = PeftModel.from_pretrained(model, instruction_model_checkpoint)


In [ ]:
model = model.merge_and_unload()

In [ ]:

pref_model_lora = get_peft_model(model, lora_config)

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"

In [ ]:
from transformers import TrainingArguments


In [ ]:
from trl import DPOTrainer, DPOConfig


In [ ]:
dpo_args = DPOConfig(
    output_dir="./tinyllama-preference-alignment",
    learning_rate=2e-5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    beta=0.1,
    report_to=[],
    logging_dir=None, # disable logging to wandb or tensorboard
    loss_type="sigmoid",  # or "hinge", depending on experiment
    remove_unused_columns=False
)

In [ ]:
trainer = DPOTrainer(
    model=pref_model_lora,
    ref_model=None,
    args=dpo_args,
    train_dataset=dataset,
    processing_class=tokenizer,   # instead of tokenizer argument
    # you can pass data_collator if needed,
    # optionally eval_dataset etc.
)


In [ ]:

trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss


TrainOutput(global_step=1, training_loss=0.6931471824645996, metrics={'train_runtime': 6.4304, 'train_samples_per_second': 0.778, 'train_steps_per_second': 0.156, 'total_flos': 4366854955008.0, 'train_loss': 0.6931471824645996})

In [ ]:
question = "Explain how Metformin works in the human body and why some researchers believe it could have benefits beyond diabetes treatment."

In [ ]:
model_path = "/content/tinyllama-preference-alignment/checkpoint-1"

In [ ]:
preference_aligned_model = AutoModelForCausalLM.from_pretrained(model_path, dtype=torch.float16)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [ ]:
preference_aligned_model.to("cuda")


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): lora.Linear(
            (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
            (lora_dropout): ModuleDict(
              (default): Dropout(p=0.05, inplace=False)
            )
            (lora_A): ModuleDict(
              (default): Linear(in_features=2048, out_features=8, bias=False)
            )
            (lora_B): ModuleDict(
              (default): Linear(in_features=8, out_features=2048, bias=False)
            )
            (lora_embedding_A): ParameterDict()
            (lora_embedding_B): ParameterDict()
            (lora_magnitude_vector): ModuleDict()
          )
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): lora.Linear(
            (base_layer): Linear(in_features=2048, out_features=256, bia

In [ ]:

inputs = tokenizer(question, return_tensors="pt").to("cuda")

In [ ]:

outputs = preference_aligned_model.generate(
    **inputs,
    max_new_tokens=200,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

In [ ]:
print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))



Model Output:

Explain how Metformin works in the human body and why some researchers believe it could have benefits beyond diabetes treatment. Learn about common side effects of this medication and their symptoms. Discover more about how Diabetes can be managed through lifestyle changes and other treatments.
Diabetic Kidney Disease, Treatment, Diabetes
The kidneys are a pair of small organs that filter blood to remove harmful substances like protein and waste from the bloodstream. The kidneys also help regulate blood sugar levels by filtering out glucose or sugar in the urine. If you have diabetes, your kidneys may not work as well as they used to. This is called kidney disease. Learn about diabetic kidney disease and the treatments available for people with this condition.


In [ ]:
!zip -r DPO_folder.zip /content/tinyllama-preference-alignment

  adding: content/tinyllama-preference-alignment/ (stored 0%)
  adding: content/tinyllama-preference-alignment/checkpoint-1/ (stored 0%)
  adding: content/tinyllama-preference-alignment/checkpoint-1/tokenizer_config.json (deflated 46%)
  adding: content/tinyllama-preference-alignment/checkpoint-1/adapter_model.safetensors (deflated 19%)
  adding: content/tinyllama-preference-alignment/checkpoint-1/trainer_state.json (deflated 56%)
  adding: content/tinyllama-preference-alignment/checkpoint-1/README.md (deflated 65%)
  adding: content/tinyllama-preference-alignment/checkpoint-1/scheduler.pt (deflated 62%)
  adding: content/tinyllama-preference-alignment/checkpoint-1/ref/ (stored 0%)
  adding: content/tinyllama-preference-alignment/checkpoint-1/ref/adapter_model.safetensors (deflated 41%)
  adding: content/tinyllama-preference-alignment/checkpoint-1/ref/adapter_config.json (deflated 56%)
  adding: content/tinyllama-preference-alignment/checkpoint-1/optimizer.pt (deflated 70%)
  adding: c

In [ ]:
from google.colab import files
files.download("DPO_folder.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>